대이터 블러오기

In [1]:
import pandas as pd
import numpy as np

raw_df = pd.read_parquet("../eda/yellow_tripdata_2026-05_snappy.parquet")
df = pd.read_parquet("../eda/data_preparation/payment_type_dataset.parquet")

In [2]:
df.head()

,trip_distance,fare_amount,trip_duration,cbd_congestion_fee,tolls_amount,PULocationID,DOLocationID,hour,day_of_week,VendorID,is_airport,payment_label
0,7.51,35.9,27.816667,0.00,0.0,138,37,0,4,2,1,현금
1,6.14,27.5,22.700000,0.75,0.0,138,237,0,4,2,1,신용카드
2,2.40,19.1,19.950000,0.75,0.0,249,232,0,4,1,0,신용카드
3,1.20,9.3,7.600000,0.75,0.0,232,114,0,4,1,0,신용카드
4,5.89,28.9,25.333333,0.00,0.0,255,236,0,4,2,0,신용카드


In [3]:
raw_df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-05-01 00:04:59,2026-05-01 00:32:48,1.0,7.51,1.0,N,138,37,2,35.9,6.00,0.5,0.00,0.0,1.0,45.40,0.0,2.0,0.00
1,2,2026-05-01 00:37:05,2026-05-01 00:59:47,1.0,6.14,1.0,N,138,237,1,27.5,6.00,0.5,9.56,0.0,1.0,49.81,2.5,2.0,0.75
2,1,2026-05-01 00:34:05,2026-05-01 00:54:02,1.0,2.40,1.0,N,249,232,1,19.1,4.25,0.5,3.73,0.0,1.0,28.58,2.5,0.0,0.75
3,1,2026-05-01 00:55:07,2026-05-01 01:02:43,0.0,1.20,1.0,N,232,114,1,9.3,4.25,0.5,3.00,0.0,1.0,18.05,2.5,0.0,0.75
4,7,2026-05-01 00:44:13,2026-05-01 00:44:13,2.0,0.86,1.0,N,140,237,1,7.2,0.00,0.5,2.44,0.0,1.0,14.64,2.5,0.0,0.00


In [4]:
#원본 데이터랑 비교하긔
import numpy as np
from scipy import stats

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

print("df :", df.shape)
print ("raw :", raw_df.shape, f"(유지율 {len(df)/len(raw_df):.2%})")
print()
df.info()

df : (3891255, 12)
raw : (4090836, 20) (유지율 95.12%)

<class 'pandas.DataFrame'>
RangeIndex: 3891255 entries, 0 to 3891254
Data columns (total 12 columns):
 #   Column              Dtype  
---  ------              -----  
 0   trip_distance       float64
 1   fare_amount         float64
 2   trip_duration       float64
 3   cbd_congestion_fee  float64
 4   tolls_amount        float64
 5   PULocationID        int32  
 6   DOLocationID        int32  
 7   hour                int32  
 8   day_of_week         int32  
 9   VendorID            int32  
 10  is_airport          int64  
 11  payment_label       str    
dtypes: float64(5), int32(5), int64(1), str(1)
memory usage: 322.0 MB


유지율 95%라서 원본 그냥 버리고 은혜님께서 진행한 데이터 믿고 변환하긔

In [5]:
print(df["payment_label"].value_counts())
print()
print(df["payment_label"].value_counts(normalize=True).round(4))

payment_label
신용카드         2660128
Flex Fare     878188
현금            352939
Name: count, dtype: int64

payment_label
신용카드         0.6836
Flex Fare    0.2257
현금           0.0907
Name: proportion, dtype: float64


In [6]:
# 컬럼별 dtype·결측·고유값 요약
summary = pd.DataFrame({
    "dtype" : df.dtypes.astype(str),
    "결측 수" : df.isna().sum(),
    "결측%" : (df.isna().mean() * 100).round(2),
    "고유값" : df.nunique(),
})
summary

,dtype,결측 수,결측%,고유값
trip_distance,float64,0,0.0,4740
fare_amount,float64,0,0.0,11401
trip_duration,float64,0,0.0,9264
cbd_congestion_fee,float64,0,0.0,2
tolls_amount,float64,0,0.0,1082
PULocationID,int32,0,0.0,259
DOLocationID,int32,0,0.0,260
hour,int32,0,0.0,24
day_of_week,int32,0,0.0,7
VendorID,int32,0,0.0,3


In [7]:
NUM_COLS = ["trip_distance", "fare_amount", "trip_duration",
            "cbd_congestion_fee", "tolls_amount"]
CAT_COLS = ["PULocationID", "DOLocationID", "hour",
            "day_of_week", "VendorID", "is_airport"]
TARGET   = "payment_label"

# 기술통계 — 평균·표준편차·분위수(25/50/75%) 포함
df[NUM_COLS].describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
trip_distance,3891255.0,3.520,4.269,0.010,1.1,1.940,3.95,199.300
fare_amount,3891255.0,21.462,17.326,0.010,10.0,16.300,26.50,1061.720
trip_duration,3891255.0,18.816,15.417,0.017,8.9,14.683,23.45,179.983
cbd_congestion_fee,3891255.0,0.501,0.353,0.000,0.0,0.750,0.75,0.750
tolls_amount,3891255.0,0.548,2.183,0.000,0.0,0.000,0.00,145.600


In [8]:
# 왜도 확인 — log1p 변환 전후 비교
pd.DataFrame({
    "skew":      df[NUM_COLS].skew().round(2),
    "skew_log1p": np.log1p(df[NUM_COLS].clip(lower=0)).skew().round(2),
})

,skew,skew_log1p
trip_distance,3.32,0.90
fare_amount,3.27,0.40
trip_duration,2.48,-0.01
cbd_congestion_fee,-0.71,-0.71
tolls_amount,5.34,3.51


서부셋 생성

잘못만들어서 이후부터는 쿨로두가 함,...ㅠ